In [1]:
json_path = 'files/FIW_family_members.json'

In [2]:
from pickle import load

embeddings_path = 'files/FIW_buffalo_sc_faces_info.pkl'
with open(embeddings_path, 'rb') as pickle_file:
    embeddings = load(pickle_file)

In [3]:
from src.family import load_families_from_json

families = load_families_from_json(json_path, embeddings_dict=embeddings)

In [4]:
import pandas as pd


family_subset_df = pd.read_csv('files/family_subset.csv', index_col=0)
train_families_set = set(family_subset_df[family_subset_df["subset"] == "train"].index)
val_families_set = set(family_subset_df[family_subset_df["subset"] == "val"].index)
test_families_set = set(family_subset_df[family_subset_df["subset"] == "test"].index)

In [5]:
import random
from tqdm import tqdm

def make_classification_tasks(families, n_others=9, not_assigning_additional=True):
    tasks = []

    all_people = []
    for k, v in families.items():
        all_people += list(v.members.values())

    if not_assigning_additional:
        all_people = [el for el in all_people if 'additional' not in el.person_id]

    for family_id, family in tqdm(families.items()):
        all_people_from_other_families = [el for el in all_people if el.family_id != family_id]
        for person_id, person_to_assign in family.members.items():
            if not_assigning_additional:
                if 'additional' in person_to_assign.person_id: #won't try to assign these
                    continue
            family_excluding_person = family.create_family_excluding_a_person(person_id)
            others_to_assign = random.sample(all_people_from_other_families, n_others)
            for person_id, family_person in family.members.items():
                if family_person != person_to_assign:
                    if 'additional' in family_person.person_id: #won't try to assign these
                        continue
                    family_person_parents = family_person.mother + family_person.father
                    family_person_children = family_person.sons + family_person.daughters
                    family_person_siblings = family_person.sisters + family_person.brothers
                    if person_to_assign in family_person_parents:
                        relation_type = 'parent'
                    elif person_to_assign in family_person_children:
                        relation_type = 'child'
                    elif person_to_assign in family_person_siblings:
                        relation_type = 'sibling'
                    else:
                        relation_type = 'other, same family'
                    family_person_excluding = family_excluding_person.members[person_id]

                    tasks.append({
                        "person to assign": person_to_assign,
                        "potential relative from the family": family_person_excluding,
                        "actual relation": relation_type,
                        "task group person to assign id": person_to_assign.person_id,
                        "task group family id": family_id})
                    for other_to_assign in others_to_assign:
                        tasks.append({
                            "person to assign": other_to_assign,
                            "potential relative from the family": family_person_excluding,
                            "actual relation": 'other_family',
                            "task group person to assign id": person_to_assign.person_id,
                            "task group family id": family_id})

    return tasks

In [6]:
train_families = {k: families[k] for k in train_families_set}
val_families = {k: families[k] for k in val_families_set}
test_families = {k: families[k] for k in test_families_set}

In [7]:
train_tasks = make_classification_tasks(train_families)
val_tasks = make_classification_tasks(val_families)
test_tasks = make_classification_tasks(test_families)

100%|██████████| 86/86 [00:00<00:00, 681.76it/s]


In [8]:
len(train_tasks), len(val_tasks), len(test_tasks)

(114400, 28260, 24480)

In [9]:
from src.feature_calculation import calculate_features

In [10]:
def make_classification_df(classification_tasks):
    index = []
    rows = []
    for classification_task in classification_tasks:
        person_to_assign = classification_task["person to assign"]
        family_person = classification_task["potential relative from the family"]
        kinship = classification_task["actual relation"]

        person_to_assign_id = person_to_assign.person_id
        person_to_assign_family_id = person_to_assign.family_id

        family_person_id = family_person.person_id
        family_id = family_person.family_id

        index.append((person_to_assign_id, person_to_assign_family_id, family_person_id, family_id))

        features = calculate_features(person_to_assign, family_person)
        features["kinship"] = kinship

        features['assignment_task'] = (classification_task['task group person to assign id'], classification_task['task group family id'])

        rows.append(features)

    df = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(index, names=["person_to_assign_id", "person_to_assign_family_id", "family_person_id", "family_id"]))
    return df


In [11]:
train_df = make_classification_df(train_tasks)
val_df = make_classification_df(val_tasks)
test_df = make_classification_df(test_tasks)

In [12]:
train_df.head()

,,,,birthplace_distance,surname_distance,birthdate_distance,person_to_assign_name_other_person_middlename_distance,other_person_name_person_to_assign_middlename_distance,gender_male,other_person_gender_male,distance,sons_distance,daughters_distance,...,grandmothers_distance,grandfathers_distance,granddaughters_distance,grandsons_distance,nieces_distance,nephews_distance,aunts_distance,uncles_distance,kinship,assignment_task
person_to_assign_id,person_to_assign_family_id,family_person_id,family_id,,,,,,,,,,,,,,,,,,,,,
F0497_1,F0497,F0497_2,F0497,None,None,None,None,None,True,False,1.062190,NaN,0.942824,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"other, same family","(F0497_1, F0497)"
F0298_4,F0298,F0497_2,F0497,None,None,None,None,None,True,False,0.927383,NaN,0.942088,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_family,"(F0497_1, F0497)"
F0228_4,F0228,F0497_2,F0497,None,None,None,None,None,False,False,0.953875,NaN,0.816106,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_family,"(F0497_1, F0497)"
F0696_6,F0696,F0497_2,F0497,None,None,None,None,None,True,False,1.086978,NaN,0.893729,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_family,"(F0497_1, F0497)"
F0013_4,F0013,F0497_2,F0497,None,None,None,None,None,False,False,1.014838,NaN,0.950367,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,other_family,"(F0497_1, F0497)"


In [13]:
train_df['kinship'].value_counts()

kinship
other_family          102960
other, same family      5163
sibling                 2134
parent                  2072
child                   2071
Name: count, dtype: int64

In [14]:
train_df['kinship'] = train_df['kinship'].apply(lambda x: 'other' if x[:5] == 'other' else x)
val_df['kinship'] = val_df['kinship'].apply(lambda x: 'other' if x[:5] == 'other' else x)
test_df['kinship'] = test_df['kinship'].apply(lambda x: 'other' if x[:5] == 'other' else x)

In [15]:
X_train = train_df.drop(columns=['kinship', 'assignment_task'], axis=1)
y_train = train_df['kinship']

X_validation = val_df.drop(columns=['kinship', 'assignment_task'], axis=1)
y_validation = val_df['kinship']

In [16]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [17]:
from sklearn.metrics import classification_report


val_preds = rf.predict(X_validation)
print(classification_report(y_validation, val_preds))

              precision    recall  f1-score   support

       child       0.74      0.51      0.61       486
       other       0.97      0.99      0.98     26746
      parent       0.53      0.12      0.19       486
     sibling       0.75      0.52      0.62       542

    accuracy                           0.96     28260
   macro avg       0.75      0.54      0.60     28260
weighted avg       0.95      0.96      0.95     28260



In [18]:
from sklearn.metrics import log_loss
from itertools import product

results = []

param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

# All combinations of params
for n_estimators, max_depth, min_samples_split in product(
    param_grid["n_estimators"],
    param_grid["max_depth"],
    param_grid["min_samples_split"]
):
    print(n_estimators, max_depth, min_samples_split)
    # Create model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        class_weight='balanced',
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Evaluate
    y_pred_proba = model.predict_proba(X_validation)
    loss = log_loss(y_validation, y_pred_proba)

    # Store result
    results.append({
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "min_samples_split": min_samples_split,
        "log_loss": loss
    })

results_df = pd.DataFrame(results).sort_values("log_loss")
print(results_df)

50 5 2
50 5 5
50 10 2
50 10 5
50 None 2
50 None 5
100 5 2
100 5 5
100 10 2
100 10 5
100 None 2
100 None 5
150 5 2
150 5 5
150 10 2
150 10 5
150 None 2
150 None 5
    n_estimators  max_depth  min_samples_split  log_loss
17           150        NaN                  5  0.247296
11           100        NaN                  5  0.265112
16           150        NaN                  2  0.266922
10           100        NaN                  2  0.281891
5             50        NaN                  5  0.300568
4             50        NaN                  2  0.338330
9            100       10.0                  5  0.583572
15           150       10.0                  5  0.586211
14           150       10.0                  2  0.586565
8            100       10.0                  2  0.588479
3             50       10.0                  5  0.591009
2             50       10.0                  2  0.595436
0             50        5.0                  2  0.895707
6            100        5.0             

In [19]:
rf_best_params = RandomForestClassifier(
        n_estimators=150,
        min_samples_split=5,
        random_state=42
    )
rf_best_params.fit(X_train, y_train)

RandomForestClassifier(min_samples_split=5, n_estimators=150, random_state=42)

In [20]:
val_preds = rf_best_params.predict(X_validation)
print(classification_report(y_validation, val_preds))

              precision    recall  f1-score   support

       child       0.74      0.52      0.61       486
       other       0.97      0.99      0.98     26746
      parent       0.60      0.11      0.19       486
     sibling       0.75      0.53      0.62       542

    accuracy                           0.96     28260
   macro avg       0.76      0.54      0.60     28260
weighted avg       0.95      0.96      0.95     28260



In [21]:
X_validation.index

MultiIndex([('F0821_1', 'F0821', 'F0821_2', 'F0821'),
            ('F0545_1', 'F0545', 'F0821_2', 'F0821'),
            ('F0956_4', 'F0956', 'F0821_2', 'F0821'),
            ('F0834_1', 'F0834', 'F0821_2', 'F0821'),
            ('F0650_2', 'F0650', 'F0821_2', 'F0821'),
            ('F0694_1', 'F0694', 'F0821_2', 'F0821'),
            ('F0806_1', 'F0806', 'F0821_2', 'F0821'),
            ('F0048_9', 'F0048', 'F0821_2', 'F0821'),
            ('F0173_2', 'F0173', 'F0821_2', 'F0821'),
            ('F0650_5', 'F0650', 'F0821_2', 'F0821'),
            ...
            ('F0629_3', 'F0629', 'F0629_2', 'F0629'),
            ('F0295_6', 'F0295', 'F0629_2', 'F0629'),
            ('F0560_3', 'F0560', 'F0629_2', 'F0629'),
            ('F0892_2', 'F0892', 'F0629_2', 'F0629'),
            ('F0588_1', 'F0588', 'F0629_2', 'F0629'),
            ('F0678_1', 'F0678', 'F0629_2', 'F0629'),
            ('F0048_7', 'F0048', 'F0629_2', 'F0629'),
            ('F0128_6', 'F0128', 'F0629_2', 'F0629'),
            

In [22]:
from pickle import dump


with open("files/FIW_rf_model.pkl", "wb") as pickle_file:
    dump(rf_best_params, pickle_file)

In [23]:
test_df.to_csv("files/test_df.csv")

In [24]:
with open("files/FIW_test_families.pkl", "wb") as pickle_file:
    dump(test_families, pickle_file)